# End-to-End Demo
This notebook walks through each layer of the pipeline in sequence:
1. Raw JSON ingestion output
2. Processed Parquet data
3. Analytics outputs

In [14]:
from pyspark.sql import SparkSession
import matplotlib.pyplot as plt
import json
import os
import glob

BASE_PATH = "/Users/snehamungre/projects/crypto_market_analysis"

spark = SparkSession.builder \
    .appName("CryptoDemo") \
    .config("spark.sql.warehouse.dir", f"{BASE_PATH}/spark-warehouse") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark session started successfully")

Spark session started successfully


---
## Stage 1 — Raw Ingestion Output

In [12]:
raw_files = sorted(glob.glob(f"{BASE_PATH}/data/raw/*.json"))
latest_raw = raw_files[-1]

print(f"Most recent raw file: {os.path.basename(latest_raw)}")
print(f"Total raw snapshots accumulated: {len(raw_files)}\n")

with open(latest_raw, "r") as f:
    raw_data = json.load(f)

print(f"Number of coins in latest snapshot: {len(raw_data)}")
print("\nSample record (first coin):")
print(json.dumps(raw_data[0], indent=2))

Most recent raw file: crypto_market_data_raw_2026-06-01.json
Total raw snapshots accumulated: 6

Number of coins in latest snapshot: 100

Sample record (first coin):
{
  "id": "bitcoin",
  "symbol": "btc",
  "name": "Bitcoin",
  "image": "https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400",
  "current_price": 73090,
  "market_cap": 1464531408492,
  "market_cap_rank": 1,
  "fully_diluted_valuation": 1464531408492,
  "total_volume": 19493736887,
  "high_24h": 74001,
  "low_24h": 72977,
  "price_change_24h": -864.3301678002317,
  "price_change_percentage_24h": -1.16874,
  "market_cap_change_24h": -17523829439.054688,
  "market_cap_change_percentage_24h": -1.1824,
  "circulating_supply": 20036643.0,
  "total_supply": 20036643.0,
  "max_supply": 21000000.0,
  "ath": 126080,
  "ath_change_percentage": -42.02917,
  "ath_date": "2025-10-06T18:57:42.558Z",
  "atl": 67.81,
  "atl_change_percentage": 107687.40495,
  "atl_date": "2013-07-06T00:00:00.000Z",
  "roi": null,

---
## Stage 2 — Processed Parquet Data

In [4]:
processed_df = spark.read.parquet(f"{BASE_PATH}/data/processed")
d = "2026-06-01"

print(f"Total records in processed layer: {processed_df.count()}")
print(f"Total records in processed layer for date {d}: {processed_df[processed_df['updated_date'] == d].count()}")
print(f"Number of partitions (dates): {processed_df.select('updated_date').distinct().count()}")
print("\nSchema:")
processed_df.printSchema()

Total records in processed layer: 401
Total records in processed layer for date 2026-06-01: 98


Number of partitions (dates): 10

Schema:
root
 |-- ath: double (nullable = true)
 |-- ath_change_percentage: double (nullable = true)
 |-- ath_date: timestamp (nullable = true)
 |-- atl: double (nullable = true)
 |-- atl_change_percentage: double (nullable = true)
 |-- atl_date: timestamp (nullable = true)
 |-- circulating_supply: double (nullable = true)
 |-- current_price: double (nullable = true)
 |-- fully_diluted_valuation: long (nullable = true)
 |-- high_24h: double (nullable = true)
 |-- id: string (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- low_24h: double (nullable = true)
 |-- market_cap: long (nullable = true)
 |-- market_cap_change_24h: double (nullable = true)
 |-- market_cap_change_percentage_24h: double (nullable = true)
 |-- market_cap_rank: long (nullable = true)
 |-- max_supply: double (nullable = true)
 |-- name: string (nullable = true)
 |-- price_change_24h: double (nullable = true)
 |-- price_change_percentage_24h: double (nullable = tr

In [5]:
print("Dates available in processed layer:")
processed_df.select("updated_date").distinct().orderBy("updated_date").show(truncate=False)

print("Sample processed records:")
processed_df.select(
    "name", "current_price", "market_cap", "total_volume",
    "circulating_supply", "updated_date"
).orderBy("market_cap", ascending=False).show(15, truncate=False)

Dates available in processed layer:
+------------+
|updated_date|
+------------+
|2026-05-18  |
|2026-05-21  |
|2026-05-23  |
|2026-05-24  |
|2026-05-25  |
|2026-05-26  |
|2026-05-28  |
|2026-05-29  |
|2026-05-31  |
|2026-06-01  |
+------------+

Sample processed records:
+--------+-------------+-------------+---------------+--------------------+------------+
|name    |current_price|market_cap   |total_volume   |circulating_supply  |updated_date|
+--------+-------------+-------------+---------------+--------------------+------------+
|Bitcoin |77910.0      |1560349334869|2.9194154601E10|2.00322E7           |2026-05-21  |
|Bitcoin |77265.0      |1547282708175|2.4931599587E10|2.0034078E7         |2026-05-25  |
|Bitcoin |73306.0      |1469300843946|3.239469181E10 |2.003599E7          |2026-05-29  |
|Bitcoin |73090.0      |1464531408492|1.9493736887E10|2.0036643E7         |2026-06-01  |
|Ethereum|2140.3       |258293665100 |1.1985654099E10|1.206856184990976E8 |2026-05-21  |
|Ethereum|2110.

---
## Stage 4 — Analytics Outputs
The analytics job produces five output tables. We load and display each one below.

In [6]:
print("Average Market Cap Rankings:")
spark.read.parquet(f"{BASE_PATH}/data/analytics/avg_market_cap") \
    .orderBy("avg_market_cap_rank") \
    .show(10, truncate=False)

Average Market Cap Rankings:
+------------+------------------+-------------------+
|name        |avg_market_cap    |avg_market_cap_rank|
+------------+------------------+-------------------+
|Bitcoin     |1.5103660738705E12|1                  |
|Ethereum    |2.4855161518825E11|2                  |
|Tether      |1.8908386459825E11|3                  |
|BNB         |8.90511361235E10  |4                  |
|XRP         |8.289547838125E10 |5                  |
|USDC        |7.615694519825E10 |6                  |
|Solana      |4.85028959165E10  |7                  |
|TRON        |3.380119497225E10 |8                  |
|Figure Heloc|1.865280819975E10 |9                  |
|Dogecoin    |1.569551686825E10 |10                 |
+------------+------------------+-------------------+
only showing top 10 rows


In [7]:
print("Average Price Rankings:")
spark.read.parquet(f"{BASE_PATH}/data/analytics/avg_price") \
    .orderBy("avg_price_rank") \
    .show(10, truncate=False)

Average Price Rankings:
+------------+-------------+--------------+
|name        |average_price|avg_price_rank|
+------------+-------------+--------------+
|Bitcoin     |75392.75     |1             |
|PAX Gold    |4520.412     |2             |
|Tether Gold |4513.045     |3             |
|Ethereum    |2059.703     |4             |
|BNB         |660.743      |5             |
|Zcash       |607.52       |6             |
|Monero      |380.415      |7             |
|Bitcoin Cash|332.265      |8             |
|Bittensor   |264.825      |9             |
|OUSG        |115.343      |10            |
+------------+-------------+--------------+
only showing top 10 rows


In [8]:
print("Volume to Market Cap Ratio Rankings:")
spark.read.parquet(f"{BASE_PATH}/data/analytics/vol_market_ratio") \
    .orderBy("vol_market_rank") \
    .show(10, truncate=False)

Volume to Market Cap Ratio Rankings:
+-------------------------------------+----------------+---------------+---------------+
|name                                 |vol_market_ratio|total_volume   |vol_market_rank|
+-------------------------------------+----------------+---------------+---------------+
|USD1                                 |0.46663         |2.176174676E9  |1              |
|Dash                                 |0.39574         |2.50219036E8   |2              |
|Worldcoin                            |0.33133         |4.26940363E8   |3              |
|Humanity                             |0.3263          |3.94585263E8   |4              |
|Artificial Superintelligence Alliance|0.3259          |2.0458783E8    |5              |
|Tether                               |0.29158         |5.5196217773E10|6              |
|Injective                            |0.27304         |1.67709359E8   |7              |
|Stellar                              |0.26681         |1.849652503E9  |8

In [9]:
print("Current Top Coins by Price (latest snapshot):")
spark.read.parquet(f"{BASE_PATH}/data/analytics/curr_top_price") \
    .orderBy("current_price_rank") \
    .show(10, truncate=False)

Current Top Coins by Price (latest snapshot):
+-----------+-------------+-------------+---------------+---------------+------------------+------------+
|name       |current_price|market_cap   |market_cap_rank|total_volume   |current_price_rank|updated_date|
+-----------+-------------+-------------+---------------+---------------+------------------+------------+
|Bitcoin    |73306.0      |1469300843946|1              |3.239469181E10 |1                 |2026-05-29  |
|Bitcoin    |73006.0      |1462654110065|1              |4.2283794309E10|1                 |2026-05-28  |
|Bitcoin    |77910.0      |1560349334869|1              |2.9194154601E10|1                 |2026-05-21  |
|Bitcoin    |77265.0      |1547282708175|1              |2.4931599587E10|1                 |2026-05-25  |
|Bitcoin    |73090.0      |1464531408492|1              |1.9493736887E10|1                 |2026-06-01  |
|Circle USYC|1.12         |2995497586   |35             |112456.0       |1                 |2026-05-26  |


In [10]:
print("Top Performing Assets — Composite Ranking:")
spark.read.parquet(f"{BASE_PATH}/data/analytics/top_performing_assets") \
    .select(
        "top_performing_rank", "name", "avg_market_cap",
        "avg_market_cap_rank", "average_price", "avg_price_rank",
        "total_volume", "vol_market_ratio", "vol_market_rank",
        "top_performing_score"
    ) \
    .orderBy("top_performing_rank") \
    .show(10, truncate=False)

Top Performing Assets — Composite Ranking:
+-------------------+------------+------------------+-------------------+-------------+--------------+---------------+----------------+---------------+--------------------+
|top_performing_rank|name        |avg_market_cap    |avg_market_cap_rank|average_price|avg_price_rank|total_volume   |vol_market_ratio|vol_market_rank|top_performing_score|
+-------------------+------------+------------------+-------------------+-------------+--------------+---------------+----------------+---------------+--------------------+
|1                  |Ethereum    |2.4855161518825E11|2                  |2059.703     |4             |1.3848108874E10|0.05729         |44             |11.0                |
|2                  |Zcash       |1.01268456725E10  |13                 |607.52       |6             |1.135944248E9  |0.10109         |27             |13.0                |
|3                  |Bitcoin     |1.5103660738705E12|1                  |75392.75     |1    

In [ ]:
import matplotlib.pyplot as plt

In [11]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
